# Fine-tune YOLO11n — **2-class** from barcodesai Barcodes (Kaggle)

Source: **Barcodes.v1i.yolov11.zip** (barcodesai / your Roboflow export).

## Classes (app)
- `0: datamatrix` ← source `DATAMATRIX`
- `1: CODE128` ← source `CODE128`
- Drop: C25INTER, CODE39, EAN*, PDF417, QRCODE, UPC*

## Kaggle dataset to upload
Create one Dataset (e.g. `scansdk-barcodesai-ft`) with:
1. `Barcodes.v1i.yolov11.clean.zip` — **use .clean** (original Roboflow zip has duplicate `data.yaml`; Kaggle rejects it)
   Path in repo: `.tmp-barcodes-clean/Barcodes.v1i.yolov11.clean.zip`
2. `best.pt` — prefer `runs/barcode/yolo11n_960_dm_code128_v1/weights/best.pt`
   (fallback: any prior YOLO11n barcode `best.pt`)

Optional (recommended): also add
- `barcode_best_training_images.zip`
- `barcode_dataset_merged.zip` (0=EAN drop, 1=CODE128, 2=GS1→CODE128)

## Session
1. New Notebook → **GPU T4/P100**, Internet **On**
2. **Add Input** → your dataset
3. File → Upload this notebook (or paste cells)
4. Run All → download `best_dm_code128.pt` + `barcode-yolo11n-dm-code128.onnx`


## 1) GPU + install


In [ ]:
!nvidia-smi
!pip -q install -U ultralytics

import torch
import ultralytics

print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("ultralytics", ultralytics.__version__)
assert torch.cuda.is_available(), "Enable GPU: Session options → Accelerator → GPU, then Restart"


## 2) Find inputs (`best.pt` + Barcodes zip **or** extracted folder)

Kaggle often unpacks the zip → folder `Barcodes.v1i.yolov11.clean/` (that is OK).


In [ ]:
from pathlib import Path
import shutil
import zipfile

INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

print("Inputs:")
!ls -la /kaggle/input

hits = list(INPUT_ROOT.rglob("best.pt"))
assert hits, "Upload best.pt into your Kaggle Dataset, then Add Input"
DATASET_DIR = hits[0].parent
print("Dataset dir:", DATASET_DIR)
!ls -lah {DATASET_DIR}

shutil.copy2(hits[0], WORK / "best.pt")
print("Copied", hits[0], "→", WORK / "best.pt")


def looks_like_roboflow_root(p: Path) -> bool:
    return p.is_dir() and ((p / "train" / "images").exists() or any(p.rglob("train/images")))


def find_barcodes_source() -> tuple[str, Path]:
    """Return ('zip'|'dir', path). Kaggle often unpacks .zip into a folder."""
    zip_patterns = [
        "Barcodes.v1i.yolov11.clean.zip",
        "Barcodes.v1i.yolov11.zip",
        "Barcodes.v1i.yolov8.zip",
        "barcodes.v1i.yolov11.zip",
    ]
    for name in zip_patterns:
        for p in INPUT_ROOT.rglob(name):
            if p.is_file():
                return "zip", p

    # Prefer extracted folder names from clean upload
    dir_names = [
        "Barcodes.v1i.yolov11.clean",
        "Barcodes.v1i.yolov11",
        "barcodes.v1i.yolov11.clean",
    ]
    for name in dir_names:
        for p in INPUT_ROOT.rglob(name):
            if looks_like_roboflow_root(p):
                return "dir", p

    # Any *Barcode* folder with train/images
    for p in sorted(INPUT_ROOT.rglob("*")):
        if not p.is_dir():
            continue
        n = p.name.lower()
        if "barcode" not in n:
            continue
        if "merged" in n or "best_training" in n:
            continue
        if looks_like_roboflow_root(p):
            return "dir", p

    # Fallback: any remaining Barcode*.zip
    cands = [
        p
        for p in list(INPUT_ROOT.rglob("*Barcode*.zip")) + list(INPUT_ROOT.rglob("*barcode*.zip"))
        if p.is_file() and "merged" not in p.name.lower() and "best_training" not in p.name.lower()
    ]
    if cands:
        clean = [p for p in cands if "clean" in p.name.lower()]
        return "zip", max(clean or cands, key=lambda p: p.stat().st_size)

    raise AssertionError(
        "Barcodes source not found. Expected zip or folder "
        "'Barcodes.v1i.yolov11.clean' with train/images under /kaggle/input"
    )


kind, BARCODES_SRC = find_barcodes_source()
print("Barcodes source:", kind, BARCODES_SRC)
if kind == "zip":
    print("size_MB", round(BARCODES_SRC.stat().st_size / 1e6, 1))
else:
    print("train/images?", (BARCODES_SRC / "train" / "images").exists())
Path(WORK / "BARCODES_KIND.txt").write_text(kind)
Path(WORK / "BARCODES_SRC.txt").write_text(str(BARCODES_SRC))
print("OK")


## 3) Unpack Barcodes → filter to 2-class YOLO layout

Roboflow layout `train|valid|test/{images,labels}` → `images/{train,val,test}` + remap classes.


In [ ]:
from pathlib import Path
import shutil
import zipfile
from collections import Counter

WORK = Path("/kaggle/working")
kind = (WORK / "BARCODES_KIND.txt").read_text().strip()
BARCODES_SRC = Path((WORK / "BARCODES_SRC.txt").read_text().strip())

RAW = WORK / "barcodesai_raw"
OUT = WORK / "combined_2class_dm_code128"

if RAW.exists():
    shutil.rmtree(RAW)
if OUT.exists():
    shutil.rmtree(OUT)

# find root that has train/images
def find_roboflow_root(root: Path) -> Path:
    if (root / "train" / "images").exists():
        return root
    hits = list(root.rglob("train/images"))
    if hits:
        return hits[0].parent.parent
    raise SystemExit(f"Could not find train/images under {root}")


if kind == "zip":
    RAW.mkdir(parents=True)
    print("Extracting zip…", BARCODES_SRC)
    with zipfile.ZipFile(BARCODES_SRC, "r") as zf:
        zf.extractall(RAW)
    src_root = find_roboflow_root(RAW)
else:
    # Already extracted on Kaggle (e.g. Barcodes.v1i.yolov11.clean/)
    print("Using extracted folder…", BARCODES_SRC)
    src_root = find_roboflow_root(BARCODES_SRC)

print("Roboflow root:", src_root)

# barcodesai names: 0 C25INTER, 1 CODE128, 2 CODE39, 3 DATAMATRIX, ...
CLASS_MAP = {
    1: 1,  # CODE128 → CODE128
    3: 0,  # DATAMATRIX → datamatrix
}

SPLIT_MAP = {"train": "train", "valid": "val", "val": "val", "test": "test"}
VALID_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
NL = chr(10)

for split in ("train", "val", "test"):
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

img_n = Counter()
box_n = Counter()
skipped = 0

for src_split, dst_split in SPLIT_MAP.items():
    img_dir = src_root / src_split / "images"
    lbl_dir = src_root / src_split / "labels"
    if not img_dir.exists():
        continue
    for img in sorted(img_dir.iterdir()):
        if img.suffix.lower() not in VALID_EXT:
            continue
        lbl = lbl_dir / f"{img.stem}.txt"
        out_lines = []
        if lbl.exists():
            for line in lbl.read_text().strip().splitlines():
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                cid = int(float(parts[0]))
                if cid not in CLASS_MAP:
                    continue
                out_lines.append(" ".join([str(CLASS_MAP[cid])] + parts[1:]))
                box_n[CLASS_MAP[cid]] += 1
        if not out_lines:
            skipped += 1
            continue
        stem = f"bai_{img.stem}"
        shutil.copy2(img, OUT / "images" / dst_split / f"{stem}{img.suffix.lower()}")
        (OUT / "labels" / dst_split / f"{stem}.txt").write_text(NL.join(out_lines) + NL)
        img_n[dst_split] += 1

print("Barcodesai images:", dict(img_n))
print("Barcodesai boxes:", dict(sorted(box_n.items())), "(0=datamatrix, 1=CODE128)")
print("Skipped (no DM/CODE128):", skipped)
assert img_n.get("train", 0) > 0 and img_n.get("val", 0) > 0
assert box_n.get(0, 0) > 0 and box_n.get(1, 0) > 0
Path(WORK / "COMBINED_ROOT.txt").write_text(str(OUT))
print("OK →", OUT)


## 4) Optional — merge your older DM + CODE128/GS1 sets

Runs only if zips/folders are present in the Kaggle input.


In [ ]:
from pathlib import Path
import shutil
import zipfile
from collections import Counter

INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
OUT = Path((WORK / "COMBINED_ROOT.txt").read_text().strip())
VALID_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
NL = chr(10)


def find_asset(stem: str):
    for z in INPUT_ROOT.rglob(f"{stem}.zip"):
        return "zip", z
    for d in INPUT_ROOT.rglob(stem):
        if d.is_dir():
            return "dir", d
    return None, None


def unwrap(root: Path) -> Path:
    if (root / "images").exists():
        return root
    nested = list(root.rglob("images/train"))
    if nested:
        return nested[0].parent.parent
    return root


def stage(stem: str, dest: Path):
    kind, src = find_asset(stem)
    if src is None:
        return None
    print(f"Merging {stem}: {kind} → {src}")
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    if kind == "zip":
        with zipfile.ZipFile(src, "r") as zf:
            zf.extractall(dest)
    else:
        shutil.copytree(src, dest, dirs_exist_ok=True)
    return unwrap(dest)


def remap_copy(root: Path, class_map: dict, prefix: str):
    if root is None or not (root / "images").exists():
        print(f"Skip {prefix}: not found")
        return Counter()
    n = Counter()
    for split in ("train", "val", "test"):
        img_dir = root / "images" / split
        lbl_dir = root / "labels" / split
        if not img_dir.exists():
            continue
        for img in sorted(img_dir.iterdir()):
            if img.suffix.lower() not in VALID_EXT:
                continue
            lbl = lbl_dir / f"{img.stem}.txt"
            lines = []
            if lbl.exists():
                for line in lbl.read_text().strip().splitlines():
                    parts = line.split()
                    if len(parts) < 5:
                        continue
                    cid = int(float(parts[0]))
                    if cid not in class_map:
                        continue
                    lines.append(" ".join([str(class_map[cid])] + parts[1:]))
            if not lines:
                continue
            stem = f"{prefix}_{img.stem}"
            shutil.copy2(img, OUT / "images" / split / f"{stem}{img.suffix.lower()}")
            (OUT / "labels" / split / f"{stem}.txt").write_text(NL.join(lines) + NL)
            n[split] += 1
    print(f"{prefix} added:", dict(n))
    return n


dm_root = stage("barcode_best_training_images", WORK / "dm_ds")
bc_root = stage("barcode_dataset_merged", WORK / "bc_ds")

# DM: any class → 0
remap_copy(dm_root, {i: 0 for i in range(32)}, "dm")
# merged: 1 CODE128 + 2 GS1 → 1
remap_copy(bc_root, {1: 1, 2: 1}, "bc")
print("Merge done (optional assets may be skipped)")


## 5) Write `data.yaml` + verify counts


In [ ]:
from pathlib import Path
from collections import Counter

OUT = Path(Path("/kaggle/working/COMBINED_ROOT.txt").read_text().strip())

yaml_text = """path: /kaggle/working/combined_2class_dm_code128
train: images/train
val: images/val
test: images/test
nc: 2
names:
  0: datamatrix
  1: CODE128
"""
(OUT / "data.yaml").write_text(yaml_text)

img_counts = {}
box_counts = Counter()
for split in ("train", "val", "test"):
    imgs = [p for p in (OUT / "images" / split).glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}]
    img_counts[split] = len(imgs)
    for lf in (OUT / "labels" / split).glob("*.txt"):
        for line in lf.read_text().strip().splitlines():
            if line.strip():
                box_counts[int(float(line.split()[0]))] += 1

print("Images:", img_counts)
print("Boxes:", dict(sorted(box_counts.items())))
assert img_counts["train"] > 0 and img_counts["val"] > 0
assert box_counts[0] > 0 and box_counts[1] > 0
print("OK data.yaml")


## 6) Fine-tune from `best.pt`

T4 tip: if OOM, set `BATCH = 4`.


In [ ]:
from ultralytics import YOLO
from pathlib import Path

BEST_PT = Path("/kaggle/working/best.pt")
DATA_YAML = Path("/kaggle/working/combined_2class_dm_code128/data.yaml")
assert BEST_PT.exists() and DATA_YAML.exists()

PROJECT = "/kaggle/working/runs/barcode"
RUN_NAME = "yolo11n_960_dm_code128_barcodesai_v1"
IMGSZ = 960
EPOCHS = 80
BATCH = 8  # drop to 4 if CUDA OOM

model = YOLO(str(BEST_PT))
results = model.train(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    device=0,
    patience=25,
    freeze=10,
    cls=1.0,
    box=7.5,
    close_mosaic=12,
    cos_lr=True,
)
print("Done:", Path(PROJECT) / RUN_NAME / "weights")


## 7) Validate


In [ ]:
from pathlib import Path
from ultralytics import YOLO

PROJECT = Path("/kaggle/working/runs/barcode")
candidates = sorted(PROJECT.glob("**/weights/best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
assert candidates, "No best.pt"
BEST = candidates[0]
print("Using", BEST)

model = YOLO(str(BEST))
metrics = model.val(data="/kaggle/working/combined_2class_dm_code128/data.yaml", imgsz=960)
print("mAP50:", float(metrics.box.map50))
print("mAP50-95:", float(metrics.box.map))
print("Precision:", float(metrics.box.mp))
print("Recall:", float(metrics.box.mr))
try:
    print("Per-class AP50:", [float(x) for x in metrics.box.ap50])
except Exception:
    pass


## 8) Export ONNX + download links


In [ ]:
from pathlib import Path
from ultralytics import YOLO
import shutil
from IPython.display import FileLink, display

PROJECT = Path("/kaggle/working/runs/barcode")
candidates = sorted(PROJECT.glob("**/weights/best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
BEST = candidates[0]
print("Exporting", BEST)

model = YOLO(str(BEST))
model.export(format="onnx", imgsz=960, simplify=True)

onnx_src = BEST.with_name("best.onnx")
if not onnx_src.exists():
    onnx_src = BEST.parent / "best.onnx"
assert onnx_src.exists(), f"ONNX missing near {BEST}"

pt_dst = Path("/kaggle/working/best_dm_code128.pt")
onnx_dst = Path("/kaggle/working/barcode-yolo11n-dm-code128.onnx")
shutil.copy2(BEST, pt_dst)
shutil.copy2(onnx_src, onnx_dst)
print("Wrote", pt_dst, pt_dst.stat().st_size)
print("Wrote", onnx_dst, onnx_dst.stat().st_size)
print("Expected ONNX: 1 x 6 x N")

for p in (pt_dst, onnx_dst):
    display(FileLink(str(p)))


## 9) Install into ScanSDK (local)

```bash
mkdir -p runs/barcode/yolo11n_960_dm_code128_v1/weights
cp ~/Downloads/best_dm_code128.pt runs/barcode/yolo11n_960_dm_code128_v1/weights/best.pt
cp ~/Downloads/barcode-yolo11n-dm-code128.onnx public/models/barcode-yolo11n.onnx
python3 scripts/probe-yolo-classes.py
```

Hard-refresh the app and retest warehouse photos.
